# 04. 지저분한 데이터 정제 — 중고차 매물 (실전)

`legacy/Untitled.ipynb` 이 하려던 작업을 다시 쓴다. 원본은 이렇게 생겼었다.

```python
old_data = pd.read_csv("C:\\Users\\gridone\\Desktop\\차량 정보\\201901_50000km_SUV.csv", ...)

def dataSplit(stringTemp):          # 60줄짜리 문자열 인덱스 파싱
    vCarTypeIdexE = vData.find("19/")
    if vData.find("쉐보레") > -1 or vData.find("르노") > -1: ...

for temp in old_data['info']:       # 행마다 append → O(n²)
    new_data = new_data.append(pd.Series(dataSplit(temp), ...), ignore_index=True)
```

문제가 네 가지다.
1. **데이터 파일이 개인 PC 경로**에 있어 아무도 재현할 수 없다
2. `find("19/")` — 2019년식만 찾는 하드코딩. 2020년 매물이 들어오면 조용히 깨진다
3. 브랜드를 `if` 로 하나씩 나열 — 새 브랜드가 나오면 빈 값
4. `DataFrame.append` — pandas 2.0 에서 **삭제됨** (지금 실행하면 AttributeError)

## 학습 목표
1. 원시 데이터의 상태를 **숫자로** 진단하기
2. 문자열 → 구조화: `str.strip`, `str.extract`(정규식 명명 그룹)
3. `to_numeric(errors="coerce")` 로 안전한 타입 변환
4. 중복·결측 처리와 **처리 전후 행 수 추적**
5. 정제 파이프라인을 함수로 묶고 `assert` 로 지키기
6. 결과를 **정답 규칙과 대조해 자기 채점**하기

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "nbtools").is_dir())
sys.path.insert(0, str(ROOT))

from nbtools import ensure_data  # noqa: E402
from nbtools.data import BRANDS  # noqa: E402  (데이터 생성에 쓰인 '정답 규칙')

ensure_data(quiet=True)
raw = pd.read_csv(ROOT / "data" / "used_cars_raw.csv")
print(f"원시 데이터 {len(raw):,}행")
raw.head()

원시 데이터 407행


,type,info,price,service
0,중고차,현대 투싼 21/06 9.1만km 디젤,"1,159만원",단순수리
1,중고차,테슬라 모델Y 22/09 4.8만km 전기,"3,685만원",-
2,중고차,제네시스 G80 16/12 14.9만km 하이브리드,"1,287만원",단순수리
3,중고차,기아 K3 20/11 12.0만km 디젤,"1,061만원",무사고
4,중고차,BMW 520i 15/12 14.6만km LPG,961만원,단순수리


## 1. 진단 — 무엇이 얼마나 더러운가

정제를 시작하기 전에 **문제의 크기를 센다.** 감으로 고치기 시작하면 무엇을 고쳤는지 모른다.

In [2]:
report = {
    "전체 행": len(raw),
    "완전 중복 행": int(raw.duplicated().sum()),
    "info 결측": int(raw["info"].isna().sum()),
    "info 공백 오염": int(raw["info"].fillna("").ne(raw["info"].fillna("").str.strip()).sum()),
    "price 숫자변환 실패": int(
        pd.to_numeric(raw["price"].astype(str).str.replace(r"[^\d]", "", regex=True), errors="coerce").isna().sum()
    ),
}
for key, value in report.items():
    print(f"  {key:<20}{value:>6,}")

print("\ndtypes:")
print(raw.dtypes)

  전체 행                   407
  완전 중복 행                  7
  info 결측                 16
  info 공백 오염              23
  price 숫자변환 실패           21

dtypes:
type       str
info       str
price      str
service    str
dtype: object


`price` 가 숫자가 아니라 **문자열 컬럼**이다. `"1,850만원"` 처럼 쉼표와 단위가 붙어 있고,
일부는 `"상담"` 이다. 이 상태로 `mean()` 을 부르면 에러가 나거나 엉뚱한 값이 나온다.

> **버전 메모**: pandas 3.0 부터 문자열 컬럼의 dtype 이 `object` 에서 **`str`** 로 바뀌었다.
> 그래서 `select_dtypes(include="object")` 로 문자열 컬럼을 고르던 코드는 3.0 에서
> 경고(`Pandas4Warning`)를 내고 언젠가 아무것도 선택하지 않게 된다.
> 버전에 상관없이 안전한 방법은 **대상 컬럼을 명시**하는 것이다.

In [3]:
raw["price"].value_counts().head(8)

price
문의         13
상담          8
1,120만원     3
2,565만원     2
1,542만원     2
1,408만원     2
901만원       2
830만원       2
Name: count, dtype: int64

## 2. 1단계 — 공백·중복 정리

순서가 중요하다. **공백을 먼저 없애야** 중복이 제대로 잡힌다
(`"현대 아반떼"` 와 `"  현대 아반떼 "` 는 다른 문자열이다).

In [4]:
TEXT_COLUMNS = ["type", "info", "price", "service"]   # 명시 — 버전·데이터 변화에 안 흔들린다

step1 = raw.copy()
step1[TEXT_COLUMNS] = step1[TEXT_COLUMNS].apply(lambda s: s.str.strip())

before = len(step1)
step1 = step1.drop_duplicates()
print(f"공백 제거 후 중복 제거: {before:,} → {len(step1):,}행 ({before - len(step1)}행 제거)")

공백 제거 후 중복 제거: 407 → 400행 (7행 제거)


## 3. 2단계 — 가격을 숫자로

`errors="coerce"` 는 변환할 수 없는 값을 예외 대신 `NaN` 으로 만든다.
**그리고 몇 개가 NaN 이 됐는지 반드시 센다** — 조용한 데이터 손실을 막는 유일한 방법이다.

In [5]:
step2 = step1.copy()
step2["price_manwon"] = pd.to_numeric(
    step2["price"].str.replace(",", "", regex=False).str.replace("만원", "", regex=False),
    errors="coerce",
)

failed = step2["price_manwon"].isna().sum()
print(f"숫자 변환 실패 {failed}건 ({failed / len(step2) * 100:.1f}%)")
print("실패한 값 예시:", step2.loc[step2["price_manwon"].isna(), "price"].unique().tolist()[:5])

숫자 변환 실패 21건 (5.2%)
실패한 값 예시: ['상담', '문의']


가격이 없는 매물은 가격 분석에서 **제외**한다. 0 으로 채우면 평균이 무너진다.

In [6]:
step2 = step2.dropna(subset=["price_manwon"])
step2["price_manwon"] = step2["price_manwon"].astype(int)
print(f"가격 있는 매물 {len(step2):,}행")

가격 있는 매물 379행


## 4. 3단계 — 문자열에서 구조 뽑아내기 (정규식)

`info` 는 `"현대 그랜저 19/03 4.1만km 가솔린"` 같은 한 덩어리다. 원본처럼 `find()` 와 `if` 로
자르는 대신, **명명 그룹을 가진 정규식 하나**로 한 번에 뽑는다.

In [7]:
PATTERN = re.compile(
    r"^(?P<brand>\S+)\s+"                 # 브랜드: 첫 단어
    r"(?P<model>.+?)\s+"                  # 모델: 최소 매칭 (뒤 패턴이 나올 때까지)
    r"(?P<year2>\d{2})/(?P<month>\d{2})\s+"   # 연식: 19/03
    r"(?P<distance>[\d.]+)만km\s+"        # 주행거리: 4.1만km
    r"(?P<fuel>\S+)$"                     # 연료: 마지막 단어
)

print(PATTERN.match("현대 그랜저 IG 2.4 19/03 4.1만km 가솔린").groupdict())

{'brand': '현대', 'model': '그랜저 IG 2.4', 'year2': '19', 'month': '03', 'distance': '4.1', 'fuel': '가솔린'}


In [8]:
step3 = step2.copy()
extracted = step3["info"].str.extract(PATTERN)
print("추출 실패(전부 NaN) 행:", int(extracted.isna().all(axis=1).sum()))
step3 = pd.concat([step3, extracted], axis=1).dropna(subset=["brand"])
step3.head(3)[["info", "brand", "model", "year2", "distance", "fuel"]]

추출 실패(전부 NaN) 행: 16


,info,brand,model,year2,distance,fuel
0,현대 투싼 21/06 9.1만km 디젤,현대,투싼,21,9.1,디젤
1,테슬라 모델Y 22/09 4.8만km 전기,테슬라,모델Y,22,4.8,전기
2,제네시스 G80 16/12 14.9만km 하이브리드,제네시스,G80,16,14.9,하이브리드


`str.extract` 는 명명 그룹을 **그대로 컬럼명**으로 만들어 준다. 60줄짜리 `dataSplit` 이 정규식 6줄로 줄었고,
새 브랜드가 나와도 자동으로 잡힌다(원본은 `if` 목록에 없으면 빈 값이었다).

## 5. 4단계 — 타입 정리와 파생 컬럼

두 자리 연식(`19`, `05`)을 네 자리로 펴는 규칙이 필요하다. 여기서는 **미래 연식은 없다**는
가정으로 처리한다. 이런 가정은 반드시 코드 옆에 적어 둔다.

In [9]:
CURRENT_YEAR = 2025

clean = step3.copy()
clean["year"] = 2000 + clean["year2"].astype(int)
clean.loc[clean["year"] > CURRENT_YEAR, "year"] -= 100      # 미래 연식이면 1900년대
clean["age"] = CURRENT_YEAR - clean["year"]
clean["distance_km"] = (clean["distance"].astype(float) * 10_000).astype(int)
clean["brand"] = clean["brand"].astype("category")
clean["fuel"] = clean["fuel"].astype("category")

clean = clean[["brand", "model", "year", "age", "distance_km", "fuel", "price_manwon", "service"]]
clean.info()

<class 'pandas.DataFrame'>
Index: 363 entries, 0 to 406
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   brand         363 non-null    category
 1   model         363 non-null    str     
 2   year          363 non-null    int64   
 3   age           363 non-null    int64   
 4   distance_km   363 non-null    int64   
 5   fuel          363 non-null    category
 6   price_manwon  363 non-null    int64   
 7   service       363 non-null    str     
dtypes: category(2), int64(4), str(2)
memory usage: 20.7 KB


`category` 로 바꾸면 메모리가 줄고 `groupby` 가 빨라진다. 값 종류가 적고 반복되는 컬럼의 기본 선택이다.

In [10]:
print(f"object 로 뒀을 때 brand 메모리: {step3['brand'].astype(object).memory_usage(deep=True):,} bytes")
print(f"category 로 바꾼 뒤        : {clean['brand'].memory_usage(deep=True):,} bytes")

object 로 뒀을 때 brand 메모리: 30,785 bytes
category 로 바꾼 뒤        : 3,878 bytes


## 6. 검증 — 정제가 제대로 됐는가

눈으로 훑는 대신 **불변 조건을 코드로 박아 둔다.** 데이터가 바뀌어도 자동으로 걸린다.

In [11]:
assert clean["price_manwon"].between(50, 30_000).all(), "가격이 상식 범위를 벗어난 행이 있다"
assert clean["year"].between(1990, CURRENT_YEAR).all(), "연식 이상"
assert clean["age"].ge(0).all(), "차령이 음수"
assert clean.notna().all().all(), "결측이 남아 있다"
print(f"검증 통과 — 최종 {len(clean):,}행 (원시 {len(raw):,}행에서 {len(raw) - len(clean):,}행 제외)")

clean.describe().round(1)

검증 통과 — 최종 363행 (원시 407행에서 44행 제외)


,year,age,distance_km,price_manwon
count,363.0,363.0,363.0,363.0
mean,2019.4,5.6,85220.4,1813.3
std,2.8,2.8,62856.3,1204.9
min,2015.0,1.0,2000.0,329.0
25%,2017.0,3.0,31000.0,976.5
50%,2019.0,6.0,69000.0,1433.0
75%,2022.0,8.0,129000.0,2179.0
max,2024.0,10.0,220000.0,5576.0


## 7. 자기 채점 — 원래 규칙을 되찾았는가

이 데이터는 `nbtools/data.py` 가 **브랜드 등급 × 연식 감가 × 주행거리 패널티** 규칙으로 만들었다.
정제가 제대로 됐다면 분석 결과가 그 규칙을 되짚어야 한다.

In [12]:
brand_mean = clean.groupby("brand", observed=True)["price_manwon"].mean()
truth_grade = pd.Series({name: grade for name, (grade, _) in BRANDS.items()})
compare = pd.DataFrame({"평균가격": brand_mean.round(0), "생성규칙_등급": truth_grade}).dropna()
compare["가격순위"] = compare["평균가격"].rank(ascending=False).astype(int)
compare["등급순위"] = compare["생성규칙_등급"].rank(ascending=False).astype(int)
compare = compare.sort_values("가격순위")

correlation = compare["평균가격"].corr(compare["생성규칙_등급"])
print(f"평균가격 ↔ 브랜드 등급 상관계수: {correlation:.3f}")
compare

평균가격 ↔ 브랜드 등급 상관계수: 0.966


,평균가격,생성규칙_등급,가격순위,등급순위
벤츠,3064.0,2.30,1,1
테슬라,2686.0,2.05,2,3
제네시스,2268.0,1.85,3,4
BMW,2102.0,2.10,4,2
현대,1236.0,1.00,5,5
기아,1136.0,0.97,6,6
쉐보레,1035.0,0.85,7,7
르노,968.0,0.83,8,8


상관 0.9 이상이면 정제와 집계가 신뢰할 만하다는 뜻이다. 실무에서는 정답 규칙이 없지만,
**"이미 아는 사실을 데이터가 재현하는가"**(작년 매출 총합, 알려진 비율 등)를 검산으로 삼는 습관은 같다.

In [13]:
# 연식·주행거리와의 관계도 규칙대로인지 확인
print("차령과 가격 상관    :", round(clean["age"].corr(clean["price_manwon"]), 3), "(음수여야 정상)")
print("주행거리와 가격 상관:", round(clean["distance_km"].corr(clean["price_manwon"]), 3), "(음수여야 정상)")

차령과 가격 상관    : -0.653 (음수여야 정상)
주행거리와 가격 상관: -0.477 (음수여야 정상)


## 8. 파이프라인으로 묶기

노트북에서 셀 단위로 만든 절차는 **함수 하나**로 정리해 둬야 재사용된다.
(그래야 다음 달 데이터가 왔을 때 셀을 다시 순서대로 누르지 않는다.)

In [14]:
def clean_used_cars(raw: pd.DataFrame, current_year: int = CURRENT_YEAR) -> pd.DataFrame:
    """원시 매물 CSV → 분석용 DataFrame. 각 단계는 위 셀들과 1:1 대응한다."""
    df = raw.copy()
    df[TEXT_COLUMNS] = df[TEXT_COLUMNS].apply(lambda s: s.str.strip())
    df = df.drop_duplicates()

    price = df["price"].str.replace(",", "", regex=False).str.replace("만원", "", regex=False)
    df["price_manwon"] = pd.to_numeric(price, errors="coerce")
    df = df.dropna(subset=["price_manwon"])

    df = pd.concat([df, df["info"].str.extract(PATTERN)], axis=1).dropna(subset=["brand"])
    df["year"] = 2000 + df["year2"].astype(int)
    df.loc[df["year"] > current_year, "year"] -= 100
    df["age"] = current_year - df["year"]
    df["distance_km"] = (df["distance"].astype(float) * 10_000).astype(int)

    return df.assign(
        price_manwon=df["price_manwon"].astype(int),
        brand=df["brand"].astype("category"),
        fuel=df["fuel"].astype("category"),
    )[["brand", "model", "year", "age", "distance_km", "fuel", "price_manwon", "service"]].reset_index(drop=True)


rebuilt = clean_used_cars(raw)
print("셀 단위 결과와 동일한가:", rebuilt.equals(clean.reset_index(drop=True)))

셀 단위 결과와 동일한가: True


In [15]:
output = ROOT / "data" / "used_cars_clean.csv"
rebuilt.to_csv(output, index=False, encoding="utf-8-sig")   # utf-8-sig: 엑셀에서 한글이 안 깨진다
print(f"저장: data/{output.name} ({output.stat().st_size:,} bytes)")

저장: data/used_cars_clean.csv (18,162 bytes)


## 정리

| 원본 방식 | 지금 방식 |
|---|---|
| 개인 PC 절대경로 | 코드로 생성하는 `data/` |
| `find("19/")` 하드코딩 | 정규식 명명 그룹 |
| 브랜드 `if` 나열 | 패턴이 자동으로 추출 |
| `append` 루프 (O(n²), 2.0에서 삭제) | 벡터화된 `str.extract` |
| 검증 없음 | `assert` 불변 조건 + 정답 규칙 대조 |
| 셀에 흩어진 절차 | `clean_used_cars()` 함수 |

다음: **05. 시각화** — 정제한 데이터를 그림으로.